# Ethan's Simple Wellness Assistant

A beginner-friendly AI **agent** that helps you:
- Track food and calories
- Log workouts
- Get meal suggestions
- See your daily summary

Everything is saved to a CSV file you can open in Excel!

> **Built with LangGraph + LangChain + Gemini**  
> **Observability powered by LangSmith** — watch [LangSmith 101 by James Briggs](https://www.youtube.com/watch?v=Iyc80hY2yYk) for a full walkthrough of everything this notebook uses.


## Step 1 — Install Packages

Run this cell once.

In [2]:
# Upgrade LangChain packages safely
# pandas is pinned to 2.2.2 — Colab's built-in tools require exactly this version
!pip install -q -U langchain langgraph langchain-google-genai python-dotenv langsmith
!pip install -q "pandas==2.2.2"

print("✅ All packages installed!")

✅ All packages installed!


## Step 2 — Load API Keys from Google Drive

Your `.env` file at `/content/drive/MyDrive/ai_agents/.env` must contain:
```
GEMINI_API_KEY=your_gemini_key_here
LANGSMITH_API_KEY=your_langsmith_key_here
```

Get a free LangSmith key at [smith.langchain.com](https://smith.langchain.com)


In [1]:
import os
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

ENV_PATH = "/content/drive/MyDrive/ai_agents/.env"
load_dotenv(ENV_PATH, override=True)

for key in ["GEMINI_API_KEY", "LANGSMITH_API_KEY"]:
    os.environ.pop(key, None)
load_dotenv(ENV_PATH, override=True)

GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

assert GEMINI_API_KEY,    "GEMINI_API_KEY not found in .env"
assert LANGSMITH_API_KEY, "LANGSMITH_API_KEY not found in .env"

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# ── LangSmith tracing (James Briggs LangSmith 101) ───────────────────────────
# Setting these four variables is all LangChain/LangGraph needs to send
# every agent run to LangSmith automatically — no code changes required.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]   = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"]    = LANGSMITH_API_KEY
os.environ["LANGCHAIN_PROJECT"]    = "wellness-agent"

print("✅ Keys loaded!")
print(f"Gemini key    ends in: ...{GEMINI_API_KEY[-4:]}")
print(f"LangSmith key ends in: ...{LANGSMITH_API_KEY[-4:]}")
print("🔍 LangSmith tracing is ON  →  visit smith.langchain.com → wellness-agent")

Mounted at /content/drive


AssertionError: GEMINI_API_KEY not found in .env

## Step 3 — Imports

In [ ]:
from datetime import datetime
import csv
from pathlib import Path

import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent   # moved to langgraph.prebuilt in LangChain v0.2+
from langsmith import traceable, Client             # traceable wraps any function as a traced span

print("✅ Imports ready!")

✅ Imports ready!


## Step 4 — Create the CSV Log File

In [ ]:
LOG_FILE = "my_wellness_log.csv"

if not Path(LOG_FILE).exists():
    with open(LOG_FILE, "w", newline="") as f:
        csv.writer(f).writerow(["date", "type", "name", "calories", "protein", "carbs", "fat"])
    print(f"✅ Created: {LOG_FILE}")
else:
    print(f"✅ Log file already exists: {LOG_FILE}")

✅ Created: my_wellness_log.csv


## Step 5 — Define Tools with `@traceable` Inner Functions

Each `@tool` is what the **agent** calls.  
The `@traceable` decorator (from `langsmith`) wraps the *implementation* so that the
internal logic shows up as its own named span inside the LangSmith trace.

This is the key insight from James Briggs' LangSmith 101 — `@traceable` lets you add
visibility to **any** Python function, not just LangChain objects.


In [ ]:
# ── @traceable helpers — these show up as named child spans in LangSmith ────

@traceable(name="CSV: write_food_entry")
def _write_food(name, calories, protein, carbs, fat):
    today = datetime.now().strftime("%Y-%m-%d")
    with open(LOG_FILE, "a", newline="") as f:
        csv.writer(f).writerow([today, "food", name, calories, protein, carbs, fat])
    return f"Wrote food row: {name}, {calories} cal"

@traceable(name="CSV: write_workout_entry")
def _write_workout(name, calories_burned):
    today = datetime.now().strftime("%Y-%m-%d")
    with open(LOG_FILE, "a", newline="") as f:
        csv.writer(f).writerow([today, "workout", name, -calories_burned, 0, 0, 0])
    return f"Wrote workout row: {name}, -{calories_burned} cal"

@traceable(name="CSV: read_daily_totals")
def _read_daily_totals():
    today = datetime.now().strftime("%Y-%m-%d")
    food_cal = workout_cal = protein = 0
    with open(LOG_FILE, "r") as f:
        for row in csv.DictReader(f):
            if row["date"] != today:
                continue
            cals = int(float(row["calories"]))
            if row["type"] == "food":
                food_cal += cals
                protein  += int(float(row["protein"]))
            elif row["type"] == "workout":
                workout_cal += abs(cals)
    return {"food_cal": food_cal, "workout_cal": workout_cal, "protein": protein}


# ── @tool definitions — these are what the agent sees and calls ──────────────

@tool
def log_food(name: str, calories: int, protein: int = 0, carbs: int = 0, fat: int = 0) -> str:
    """Log a food item to the wellness tracker.
    Use this whenever the user mentions eating something.
    Estimate calories and macros if the user does not give exact numbers.
    """
    _write_food(name, calories, protein, carbs, fat)
    return f"✅ Logged food: {name} | {calories} cal | {protein}g protein | {carbs}g carbs | {fat}g fat"


@tool
def log_workout(name: str, calories_burned: int) -> str:
    """Log a workout to the wellness tracker.
    Use this whenever the user mentions exercising or any physical activity.
    Estimate calories burned from the activity type and duration if not given.
    """
    _write_workout(name, calories_burned)
    return f"✅ Logged workout: {name} | burned {calories_burned} cal"


@tool
def get_daily_summary() -> str:
    """Return today's total calories consumed, calories burned from workouts, and net calories.
    Use this when the user asks for a summary, their totals, or how they are doing today.
    """
    totals = _read_daily_totals()
    net = totals["food_cal"] - totals["workout_cal"]
    return (
        "📊 Today's summary:\n"
        f"  Food:    {totals['food_cal']} cal\n"
        f"  Workout: -{totals['workout_cal']} cal\n"
        f"  Net:     {net} cal\n"
        f"  Protein: {totals['protein']}g"
    )


@tool
def suggest_meal(calorie_target: int, preference: str = "balanced") -> str:
    """Suggest meals that fit the user's calorie budget.
    Use this when the user asks for meal ideas or food recommendations.
    preference options: balanced, high-protein, low-carb, vegetarian.
    """
    meals = {
        "balanced":     [("Grilled chicken + rice + broccoli", 550),
                         ("Turkey sandwich + apple",           420),
                         ("Salmon + quinoa + salad",           480)],
        "high-protein": [("Chicken breast + eggs + spinach",  480),
                         ("Greek yogurt + almonds + berries",  380),
                         ("Tuna wrap + cottage cheese",        420)],
        "low-carb":     [("Steak + cauliflower rice",          520),
                         ("Egg salad lettuce wraps",           320),
                         ("Shrimp stir-fry (no noodles)",      350)],
        "vegetarian":   [("Lentil soup + whole grain bread",   450),
                         ("Tofu scramble + avocado toast",     430),
                         ("Black bean bowl + salsa",           400)],
    }
    options = meals.get(preference.lower(), meals["balanced"])
    suggestions = [f"• {n} (~{c} cal)" for n, c in options if c <= calorie_target + 100]
    if not suggestions:
        suggestions = [f"• {options[0][0]} (~{options[0][1]} cal)"]
    return f"🍽️ Meal suggestions ({preference}, ~{calorie_target} cal target):\n" + "\n".join(suggestions)


TOOLS = [log_food, log_workout, get_daily_summary, suggest_meal]
print(f"✅ {len(TOOLS)} tools registered: {[t.name for t in TOOLS]}")
print("🔍 Inner functions decorated with @traceable will appear as child spans in LangSmith")

✅ 4 tools registered: ['log_food', 'log_workout', 'get_daily_summary', 'suggest_meal']
🔍 Inner functions decorated with @traceable will appear as child spans in LangSmith


## Step 6 — Build the Agent

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)

SYSTEM_PROMPT = (
    "You are a friendly and encouraging wellness assistant. "
    "Help the user track food, workouts, and progress toward their health goals. "
    "Always use the available tools to log entries — never guess from memory. "
    "If the user mentions eating something, call log_food with your best calorie/macro estimate. "
    "If the user mentions exercise, call log_workout with your best calorie-burn estimate. "
    "Be supportive, specific, and concise."
)

agent = create_react_agent(
    model=llm,
    tools=TOOLS,
    prompt=SYSTEM_PROMPT,
)

print("✅ Wellness agent ready!")
print("🔍 Every run → smith.langchain.com → wellness-agent project")

✅ Wellness agent ready!
🔍 Every run → smith.langchain.com → wellness-agent project


/tmp/ipython-input-1066319545.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Step 7 — Try It Out

Run any cell below. After each one visit [smith.langchain.com](https://smith.langchain.com) → **wellness-agent**.

In the trace you will now see **nested spans** — the `log_food` tool node will have a
`CSV: write_food_entry` child span showing exactly what was written to disk.
That's the `@traceable` decorator in action.


In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "I just ate oatmeal with banana for breakfast"}]})
print("Assistant:", result["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 13.038071221s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "I went for a 30-minute run"}]})
print("Assistant:", result["messages"][-1].content)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Show me today's summary"}]})
print("Assistant:", result["messages"][-1].content)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Suggest a high-protein lunch under 500 calories"}]})
print("Assistant:", result["messages"][-1].content)

## Step 8 — View Your Full Log

In [ ]:
df = pd.read_csv(LOG_FILE)
print(f"Total entries: {len(df)}\n")
display(df)

print("\n=== Daily Totals ===")
display(df.groupby(["date", "type"])["calories"].sum().unstack(fill_value=0))

## Step 9 — LangSmith Evaluation Dataset

One of the most powerful features shown in James Briggs' LangSmith 101 is **evaluations**.
You create a dataset of expected inputs/outputs, run your agent against it, and LangSmith
scores the results automatically — turning ad-hoc testing into a repeatable experiment.

Here we build a small eval dataset for the wellness agent.


In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

# ── Step 9a: Create (or reuse) an evaluation dataset ─────────────────────────
DATASET_NAME = "wellness-agent-eval"

# Only create if it doesn't already exist
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Input/output pairs for evaluating the wellness agent"
    )
    client.create_examples(
        inputs=[
            {"input": "I just ate oatmeal with banana for breakfast"},
            {"input": "I went for a 30-minute run"},
            {"input": "I had a big breakfast"},
            {"input": "Show me today's summary"},
            {"input": "Suggest a vegetarian lunch under 450 calories"},
        ],
        outputs=[
            {"expected_tool": "log_food"},
            {"expected_tool": "log_workout"},
            {"expected_tool": "log_food"},
            {"expected_tool": "get_daily_summary"},
            {"expected_tool": "suggest_meal"},
        ],
        dataset_id=dataset.id,
    )
    print(f"✅ Dataset '{DATASET_NAME}' created with 5 examples")
else:
    print(f"✅ Dataset '{DATASET_NAME}' already exists — reusing it")

In [ ]:
# ── Step 9b: Define an evaluator ─────────────────────────────────────────────
# An evaluator is a function that takes (inputs, outputs, reference_outputs)
# and returns a score. This one checks whether the agent called the right tool.

def correct_tool_called(inputs, outputs, reference_outputs):
    """Returns 1 if the agent called the expected tool, 0 otherwise."""
    expected = reference_outputs.get("expected_tool", "")
    # Walk the message list looking for a tool call with the expected name
    messages = outputs.get("messages", [])
    for msg in messages:
        # Tool call messages have a 'tool_calls' attribute
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for call in msg.tool_calls:
                if call.get("name") == expected:
                    return {"score": 1, "comment": f"Called '{expected}' ✅"}
    return {"score": 0, "comment": f"Expected '{expected}' but did not find it ❌"}


print("✅ Evaluator defined")

In [ ]:
# ── Step 9c: Run the evaluation ──────────────────────────────────────────────
# evaluate() runs every dataset example through the agent, scores each one
# with our evaluator, and sends the results to LangSmith as an experiment.

def run_agent(inputs):
    """Wrapper so evaluate() can call our agent with a plain dict input."""
    return agent.invoke({"messages": [{"role": "user", "content": inputs["input"]}]})


results = evaluate(
    run_agent,
    data=DATASET_NAME,
    evaluators=[correct_tool_called],
    experiment_prefix="wellness-tool-selection",
)

print("\n✅ Evaluation complete!")
print("View results at smith.langchain.com → wellness-agent → Experiments tab")

## Step 10 — Interactive Chat Loop

In [ ]:
print("Wellness Assistant ready — type your message ('quit' to stop)\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye! Stay healthy! 💪")
        break

    result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    print(f"\nAssistant: {result['messages'][-1].content}\n")

Wellness Assistant ready — type your message ('quit' to stop)

